In [ ]:
# Cell 1 — Installs & Imports

import os
import re
import uuid
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

import anthropic
import pandas as pd

# Load API key from .env file 


load_dotenv()

api_key = os.getenv("API_KEY")

if api_key:
    print("Cell 1 loaded — ANTHROPIC_API_KEY found")
else:
    print("   ANTHROPIC_API_KEY not found — create a .env file in this folder")
    print("   Add this line:  ANTHROPIC_API_KEY=sk-ant-yourkey")

Cell 1 loaded — ANTHROPIC_API_KEY found


In [101]:
# Cell 1b — Database Setup

import sqlite3

db_file = "tpa_logs.db"

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS request_logs (
        id            INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp     TEXT,
        user_id       TEXT,
        department    TEXT,
        use_case      TEXT,
        original_input TEXT,
        clean_input   TEXT,
        pii_detected  TEXT,
        pii_count     INTEGER,
        risk_score    INTEGER,
        risk_level    TEXT,
        action        TEXT,
        input_tokens  INTEGER,
        output_tokens INTEGER,
        total_tokens  INTEGER,
        cost_usd      REAL,
        ai_response   TEXT
    )
""")

conn.commit()
print("Database ready.")
print(f"Connected to {db_file}")

Database ready.
Connected to tpa_logs.db


In [102]:
# Cell 2- Model
model = "claude-sonnet-4-5-20250929"

#Mock user
user_id = "A123"
department = "Accounting"
use_case=  "Summarize customer loan application"

excel_file = Path("usage_tracking_logs.xlsx")

client = anthropic.Anthropic(api_key = api_key)

In [103]:
 # Cell 2b — Mock Dataset

mock_requests = [
    {"user_id": "A123", "department": "Accounting",  "use_case": "Summarize loan application",  "message": "Summarize John Smith loan application. SSN 492-11-8823, email john@gmail.com"},
    {"user_id": "B456", "department": "Marketing",   "use_case": "Generate campaign copy",       "message": "Write a short email campaign for our Q4 product launch targeting small businesses"},
    {"user_id": "C789", "department": "Operations",  "use_case": "Summarize report",             "message": "Summarize the Q3 operations report. Focus on cost overruns"},
    {"user_id": "D321", "department": "Compliance",  "use_case": "Policy check",                 "message": "Is sharing customer account number 88291-4421 with a third party vendor allowed?"},
    {"user_id": "E654", "department": "Accounting",  "use_case": "Invoice review",               "message": "Review this invoice. Vendor bank account 291-881-4492 routing number 021000021"},
    {"user_id": "F987", "department": "Marketing",   "use_case": "Customer research",            "message": "What are the best strategies for retaining high value customers in a subscription business"},
    {"user_id": "G111", "department": "Operations",  "use_case": "HR request",                   "message": "Pull up employee Sarah Davis SSN 331-44-9921 performance review from last quarter"},
    {"user_id": "H222", "department": "Compliance",  "use_case": "Regulation summary",           "message": "Summarize the key points of the new CFPB regulation on AI lending decisions"},
    {"user_id": "I333", "department": "Accounting",  "use_case": "Budget analysis",              "message": "Compare our Q1 and Q2 budget performance and identify the top 3 areas of overspend"},
    {"user_id": "J444", "department": "Operations",  "use_case": "Vendor check",                 "message": "Check vendor contact robert.chen@vendor.com password access for our confidential supplier portal"},
]

print(f"{len(mock_requests)} mock requests loaded and ready to process.")

10 mock requests loaded and ready to process.


In [ ]:
# Cell 2c — Run All Mock RequestsnThrough Pipeline

for request in mock_requests:

    # Set variables for this request
    user_id    = request["user_id"]
    department = request["department"]
    use_case   = request["use_case"]
    user_input = request["message"]

    # Step 1 - Scan for PII
    pii_found, clean_input = scan_for_pii(user_input)

    # Step 2 - Risk score
    risk_score, risk_level, action, reasons = calculate_risk_score(user_input, pii_found)

    # Step 3 - Send to AI if not blocked
    if action == "BLOCKED":
        input_tokens  = 0
        output_tokens = 0
        total_cost    = 0
        ai_response   = "BLOCKED"
    else:
        resp = client.messages.create(
            model=model,
            max_tokens=256,
            messages=[{"role": "user", "content": clean_input}]
        )
        ai_response   = resp.content[0].text
        input_tokens  = resp.usage.input_tokens
        output_tokens = resp.usage.output_tokens
        total_cost    = (input_tokens / 1_000_000) * 3 + (output_tokens / 1_000_000) * 15

    # Step 4 - Log to Excel
    log_to_excel(
        user_id, department, use_case, user_input,
        clean_input, pii_found, risk_score, risk_level,
        action, input_tokens, output_tokens, total_cost, ai_response
    )

    # Step 5 - Log to SQLite
    log_to_sqlite(
        user_id, department, use_case, user_input,
        clean_input, pii_found, risk_score, risk_level,
        action, input_tokens, output_tokens, total_cost, ai_response
    )

    print(f"{user_id} | {department} | Score: {risk_score} | {action}")

print()
print("All requests processed.")

Logged to usage_tracking_logs.xlsx
Logged to SQLite: A123 | Accounting | FLAGGED
A123 | Accounting | Score: 60 | FLAGGED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: B456 | Marketing | CLEARED
B456 | Marketing | Score: 0 | CLEARED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: C789 | Operations | CLEARED
C789 | Operations | Score: 0 | CLEARED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: D321 | Compliance | CLEARED
D321 | Compliance | Score: 20 | CLEARED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: E654 | Accounting | BLOCKED
E654 | Accounting | Score: 80 | BLOCKED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: F987 | Marketing | CLEARED
F987 | Marketing | Score: 0 | CLEARED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: G111 | Operations | FLAGGED
G111 | Operations | Score: 60 | FLAGGED
Logged to usage_tracking_logs.xlsx
Logged to SQLite: H222 | Compliance | CLEARED
H222 | Compliance | Score: 0 | CLEARED
Logged to usage_tracking_logs.xlsx
Logge

In [105]:

# Cell 3 — PII Scanner
import re

def scan_for_pii(text):
    pii_found = []
    clean_text = text

    # SSN — 492-11-8823
    ssn = re.findall(r'\b\d{3}-\d{2}-\d{4}\b', text)
    if ssn:
        pii_found.append(f"SSN: {ssn}")
        clean_text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN REDACTED]', clean_text)

    # Email — john@gmail.com
    emails = re.findall(r'\b[\w.-]+@[\w.-]+\.\w+\b', text)
    if emails:
        pii_found.append(f"Email: {emails}")
        clean_text = re.sub(r'\b[\w.-]+@[\w.-]+\.\w+\b', '[EMAIL REDACTED]', clean_text)

    # Phone — 310-555-1234
    phones = re.findall(r'\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}', text)
    if phones:
        pii_found.append(f"Phone: {phones}")
        clean_text = re.sub(r'\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}', '[PHONE REDACTED]', clean_text)

    # Credit card — 4111-1111-1111-1111
    cards = re.findall(r'\b\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}\b', text)
    if cards:
        pii_found.append(f"Credit Card: {cards}")
        clean_text = re.sub(r'\b\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}\b', '[CARD REDACTED]', clean_text)

    return pii_found, clean_text

# Test message
user_input = "Summarize John Smith's loan application. SSN 492-11-8823, email john@gmail.com, phone 310-555-1234"

pii_found, clean_input = scan_for_pii(user_input)

print(" Original message:")
print(user_input)
print()
print("PII detected:")
print(pii_found if pii_found else "None found")
print()
print("Clean message ready to send:")
print(clean_input)

 Original message:
Summarize John Smith's loan application. SSN 492-11-8823, email john@gmail.com, phone 310-555-1234

PII detected:
["SSN: ['492-11-8823']", "Email: ['john@gmail.com']", "Phone: ['310-555-1234']"]

Clean message ready to send:
Summarize John Smith's loan application. SSN [SSN REDACTED], email [EMAIL REDACTED], phone [PHONE REDACTED]


In [106]:
# Cell 4 — Risk Scoring Engine

HIGH_RISK_KEYWORDS = [
    "password", "ssn", "social security", "account number",
    "routing number", "credit card", "bank account", "pin",
    "confidential", "classified", "private key", "secret"
]

def calculate_risk_score(text, pii_found):
    score = 0
    reasons = []

    # PII was detected
    if pii_found:
        score += 40
        reasons.append(f"+40 PII detected: {len(pii_found)} type(s)")

    # High risk keywords
    text_lower = text.lower()
    for keyword in HIGH_RISK_KEYWORDS:
        if keyword in text_lower:
            score += 20
            reasons.append(f"+20 High risk keyword: '{keyword}'")

    # Long message
    if len(text) > 500:
        score += 10
        reasons.append("+10 Message exceeds 500 characters")

    # After hours
    hour = datetime.now().hour
    if hour >= 18 or hour < 6:
        score += 10
        reasons.append("+10 After hours request")

    # Threshold decision
    if score >= 70:
        level  = "HIGH RISK — BLOCKED"
        action = "BLOCKED"
    elif score >= 30:
        level  = "MEDIUM RISK — FLAGGED FOR REVIEW"
        action = "FLAGGED"
    else:
        level  = "LOW RISK — CLEARED"
        action = "CLEARED"

    return score, level, action, reasons

# Run it
risk_score, risk_level, action, reasons = calculate_risk_score(user_input, pii_found)

print("=" * 50)
print("RISK ASSESSMENT REPORT")
print("=" * 50)
print(f"User:       {user_id}")
print(f"Department: {department}")
print(f"Use Case:   {use_case}")
print()
print("Scoring breakdown:")
for r in reasons:
    print(f"  {r}")
print()
print(f"Total Score: {risk_score}/100")
print(f"Decision:    {risk_level}")
print("=" * 50)

if action == "BLOCKED":
    print("\ Request blocked. Will NOT be sent to AI.")
else:
    print(f"\nProceeding with clean message...")

RISK ASSESSMENT REPORT
User:       J444
Department: Operations
Use Case:   Vendor check

Scoring breakdown:
  +40 PII detected: 3 type(s)
  +20 High risk keyword: 'ssn'

Total Score: 60/100
Decision:    MEDIUM RISK — FLAGGED FOR REVIEW

Proceeding with clean message...


In [107]:
# Cell 4b — Prompt Optimization

optimize_prompt = True

def optimize(text):
    resp = client.messages.create(
        model=model,
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": f"""Rewrite this prompt to be clearer and more token efficient. 
Keep the exact same meaning and intent. 
Remove unnecessary words. 
Do not add any explanation, just return the rewritten prompt only.

Original prompt:
{text}"""
        }]
    )
    return resp.content[0].text.strip()

if optimize_prompt:
    optimized_input = optimize(clean_input)
    print("Original clean message:")
    print(clean_input)
    print()
    print("Optimized message:")
    print(optimized_input)
    print()
    tokens_before = len(clean_input.split())
    tokens_after  = len(optimized_input.split())
    print(f"Words before: {tokens_before}")
    print(f"Words after:  {tokens_after}")
    print(f"Reduction:    {tokens_before - tokens_after} words")
else:
    optimized_input = clean_input
    print("Optimization skipped — sending message as written.")

Original clean message:
Summarize John Smith's loan application. SSN [SSN REDACTED], email [EMAIL REDACTED], phone [PHONE REDACTED]

Optimized message:
Summarize John Smith's loan application. SSN [SSN REDACTED], email [EMAIL REDACTED], phone [PHONE REDACTED]

Words before: 14
Words after:  14
Reduction:    0 words


In [108]:
# Cell 5 — Send to AI (only if cleared or flagged)

if action == "BLOCKED":
    print("Message blocked. Skipping AI call.")
    response = None
    ai_response = None

else:
    print(f"   Sending clean message to AI...")
    print(f"   Model: {model}")
    print(f"   Message: {clean_input}")
    print()

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[
            {"role": "user", "content": clean_input}
        ]
    )

    # Pull the text out of the response
    ai_response = response.content[0].text

    # Pull token counts
    input_tokens  = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    total_tokens  = input_tokens + output_tokens

    # Calculate cost
    # Sonnet pricing — $3 per million input, $15 per million output
    input_cost  = (input_tokens  / 1_000_000) * 3.00
    output_cost = (output_tokens / 1_000_000) * 15.00
    total_cost  = input_cost + output_cost

    print("=" * 50)
    print("AI RESPONSE")
    print("=" * 50)
    print(ai_response)
    print()
    print("=" * 50)
    print("TOKEN USAGE")
    print("=" * 50)
    print(f"Input tokens:  {input_tokens}")
    print(f"Output tokens: {output_tokens}")
    print(f"Total tokens:  {total_tokens}")
    print(f"Total cost:    ${total_cost:.6f}")
    print("=" * 50)

   Sending clean message to AI...
   Model: claude-sonnet-4-5-20250929
   Message: Summarize John Smith's loan application. SSN [SSN REDACTED], email [EMAIL REDACTED], phone [PHONE REDACTED]

AI RESPONSE
I'd be happy to help summarize a loan application, but I don't actually see any loan application details in your message. 

You've shared that the applicant's name is John Smith and indicated that certain personal information (SSN, email, and phone number) has been redacted, but there are no other application details such as:

- Loan amount requested
- Loan purpose
- Employment information
- Income details
- Credit history
- Assets/liabilities
- Requested loan terms

Could you provide the actual loan application information you'd like me to summarize? I'll be glad to help once I have the relevant details.

TOKEN USAGE
Input tokens:  40
Output tokens: 139
Total tokens:  179
Total cost:    $0.002205


In [109]:
# Cell 6 — Token Tracking & Excel Log

def log_to_excel(user_id, department, use_case, user_input, 
                 clean_input, pii_found, risk_score, risk_level,
                 action, input_tokens, output_tokens, total_cost, ai_response):

    new_row = {
        "timestamp":      datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user_id":        user_id,
        "department":     department,
        "use_case":       use_case,
        "original_input": user_input,
        "clean_input":    clean_input,
        "pii_detected":   str(pii_found),
        "pii_count":      len(pii_found),
        "risk_score":     risk_score,
        "risk_level":     risk_level,
        "action":         action,
        "input_tokens":   input_tokens,
        "output_tokens":  output_tokens,
        "total_tokens":   input_tokens + output_tokens,
        "cost_usd":       round(total_cost, 6),
        "ai_response":    ai_response if ai_response else "BLOCKED",
    }

    if excel_file.exists():
        df = pd.read_excel(excel_file)
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        df = pd.DataFrame([new_row])

    df.to_excel(excel_file, index=False)
    print(f"Logged to {excel_file}")
    return df

# Run it
if response:
    df = log_to_excel(
        user_id       = user_id,
        department    = department,
        use_case      = use_case,
        user_input    = user_input,
        clean_input   = clean_input,
        pii_found     = pii_found,
        risk_score    = risk_score,
        risk_level    = risk_level,
        action        = action,
        input_tokens  = input_tokens,
        output_tokens = output_tokens,
        total_cost    = total_cost,
        ai_response   = ai_response,
    )

    print()
    print("=" * 50)
    print("SESSION SUMMARY")
    print("=" * 50)
    print(f"User:          {user_id}")
    print(f"Department:    {department}")
    print(f"PII detected:  {len(pii_found)} type(s)")
    print(f"Risk score:    {risk_score}/100")
    print(f"Decision:      {action}")
    print(f"Tokens used:   {input_tokens + output_tokens}")
    print(f"Cost:          ${total_cost:.6f}")
    print(f"Log file:      {excel_file}")
    print("=" * 50)

else:
    print("No response to log — request was blocked.")

Logged to usage_tracking_logs.xlsx

SESSION SUMMARY
User:          J444
Department:    Operations
PII detected:  3 type(s)
Risk score:    60/100
Decision:      FLAGGED
Tokens used:   179
Cost:          $0.002205
Log file:      usage_tracking_logs.xlsx


In [110]:
# Cell 6b — Log to SQLite Database

def log_to_sqlite(user_id, department, use_case, user_input,
                  clean_input, pii_found, risk_score, risk_level,
                  action, input_tokens, output_tokens, total_cost, ai_response):

    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO request_logs (
            timestamp, user_id, department, use_case,
            original_input, clean_input, pii_detected, pii_count,
            risk_score, risk_level, action,
            input_tokens, output_tokens, total_tokens,
            cost_usd, ai_response
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        user_id, department, use_case,
        user_input, clean_input, str(pii_found), len(pii_found),
        risk_score, risk_level, action,
        input_tokens, output_tokens, input_tokens + output_tokens,
        round(total_cost, 6),
        ai_response if ai_response else "BLOCKED"
    ))

    conn.commit()
    conn.close()
    print(f"Logged to SQLite: {user_id} | {department} | {action}")

# Run it
if response:
    log_to_sqlite(
        user_id, department, use_case, user_input,
        clean_input, pii_found, risk_score, risk_level,
        action, input_tokens, output_tokens, total_cost, ai_response
    )

Logged to SQLite: J444 | Operations | FLAGGED


In [111]:
# Cell 6c — Query the Database

conn   = sqlite3.connect(db_file)
cursor = conn.cursor()

# Show all records
cursor.execute("SELECT user_id, department, risk_score, action, total_tokens, cost_usd FROM request_logs")
rows = cursor.fetchall()

conn.close()

print("=" * 60)
print("TPA DATABASE — ALL RECORDS")
print("=" * 60)
print(f"{'USER':<8} {'DEPARTMENT':<12} {'SCORE':<7} {'ACTION':<10} {'TOKENS':<8} {'COST'}")
print("-" * 60)

for row in rows:
    print(f"{row[0]:<8} {row[1]:<12} {row[2]:<7} {row[3]:<10} {row[4]:<8} ${row[5]:.6f}")

print("=" * 60)
print(f"Total records: {len(rows)}")

TPA DATABASE — ALL RECORDS
USER     DEPARTMENT   SCORE   ACTION     TOKENS   COST
------------------------------------------------------------
J444     Operations   70      BLOCKED    0        $0.000000
J444     Operations   70      BLOCKED    0        $0.000000
A123     Accounting   60      FLAGGED    136      $0.001668
B456     Marketing    0       CLEARED    278      $0.003906
C789     Operations   0       CLEARED    142      $0.001854
D321     Compliance   20      CLEARED    282      $0.003918
E654     Accounting   80      BLOCKED    0        $0.000000
F987     Marketing    0       CLEARED    278      $0.003906
G111     Operations   60      FLAGGED    203      $0.002733
H222     Compliance   0       CLEARED    264      $0.003660
I333     Accounting   0       CLEARED    286      $0.003930
J444     Operations   80      BLOCKED    0        $0.000000
J444     Operations   60      FLAGGED    210      $0.002670
J444     Operations   60      FLAGGED    210      $0.002670
A123     Accounti

In [112]:
import os
print(os.getcwd())

c:\Users\seles\Documents\agent-factory\src


In [113]:
# Cell - Budget Enforcement

department_budgets = {
    "Accounting":  500000,
    "Marketing":   200000,
    "Operations":  400000,
    "Compliance":  300000,
}

department_usage = {
    "Accounting":  0,
    "Marketing":   0,
    "Operations":  0,
    "Compliance":  0,
}

# Load actual usage from Excel
df_log = pd.read_excel(excel_file)

for _, row in df_log.iterrows():
    dept = row["department"]
    if dept in department_usage:
        department_usage[dept] += row["total_tokens"]

# Report
print("=" * 50)
print("DEPARTMENT BUDGET REPORT")
print("=" * 50)

for dept, budget in department_budgets.items():
    used      = department_usage[dept]
    remaining = budget - used
    pct_used  = (used / budget) * 100

    if pct_used >= 90:
        status = "CRITICAL"
    elif pct_used >= 70:
        status = "WARNING"
    else:
        status = "OK"

    print(f"{dept:<12} | Budget: {budget:>7,} | Used: {used:>6,} | Remaining: {remaining:>7,} | {pct_used:.1f}% | {status}")

print("=" * 50)

DEPARTMENT BUDGET REPORT
Accounting   | Budget: 500,000 | Used:  1,553 | Remaining: 498,447 | 0.3% | OK
Marketing    | Budget: 200,000 | Used:  2,224 | Remaining: 197,776 | 1.1% | OK
Operations   | Budget: 400,000 | Used:  1,480 | Remaining: 398,520 | 0.4% | OK
Compliance   | Budget: 300,000 | Used:  2,202 | Remaining: 297,798 | 0.7% | OK
